## Table of Contents
- [1. PII Identification & Privacy Demonstration](#1-pii-identification--privacy-demonstration)
  - [1.1 The Privacy Risk: Identifying PII in NovaCred's Data](#11-the-privacy-risk-identifying-pii-in-novacreds-data)
  - [1.2 GDPR Data Minimization](#12-gdpr-data-minimization)
  - [1.3 Pseudonymization](#13-pseudonymization)
- [2. GDPR Compliance Mapping](#2-gdpr-compliance-mapping)
  - [2.1 Violation of Article 5: Storage Limitation](#21-violation-of-article-5-storage-limitation)
  - [2.2 Violation of Article 6: Lawful Basis for Processing](#22-violation-of-article-6-lawful-basis-for-processing)
  - [2.3 Violation of Article 14: Transparency (Data Sources)](#23-violation-of-article-14-transparency-data-sources)
  - [2.4 Implications for Article 17: Right to Erasure](#24-implications-for-article-17-right-to-erasure)
- [3. EU AI Act Implications](#3-eu-ai-act-implications)
  - [3.1 High-Risk System Classification](#31-high-risk-system-classification)
  - [3.2 The Legal Reality of Disparate Impact](#32-the-legal-reality-of-disparate-impact)
  - [3.3 Mandatory Human Oversight](#33-mandatory-human-oversight)
  - [3.4 Traceability and Auditability](#34-traceability-and-auditability)
- [4. Actionable Governance Controls](#4-actionable-governance-controls)
  - [4.1 Implement Cryptographic Consent Mechanisms](#41-implement-cryptographic-consent-mechanisms)
  - [4.2 Automate Data Lifecycle & Retention](#42-automate-data-lifecycle--retention)
  - [4.3 Algorithmic Audit Trails (Traceability)](#43-algorithmic-audit-trails-traceability)
  - [4.4 Mandatory Human-in-the-Loop (Oversight)](#44-mandatory-human-in-the-loop-oversight)
  - [4.5 Ethical Feature Engineering](#45-ethical-feature-engineering)
  - [4.6 Enforce Data Lineage Tracking](#46-enforce-data-lineage-tracking)
  - [4.7 Automated Data Minimization & Anonymization](#47-automated-data-minimization--anonymization)


# 1. PII Identification & Privacy Demonstration


In [22]:
import pandas as pd
import json

# Force Pandas to display the full width of our data without wrapping
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# 1. Load the raw dataset
with open('../data/raw_credit_applications.json', 'r') as f:
    data = json.load(f)

# Transforming the JSON into a readable table
df = pd.json_normalize(data)

print("1. ORIGINAL DATA: Exposed Personally Identifiable Information (PII)\n")

# Safely identify all rubric-required PII regardless of exact naming
pii_keywords = ['name', 'email', 'ssn', 'ip', 'birth', 'dob']
pii_cols = [col for col in df.columns if any(kw in col.lower() for kw in pii_keywords)]

print(f"IDENTIFIED PII FIELDS: {pii_cols}\n")
if pii_cols:
    # Use .to_string() to completely bypass Jupyter's auto-truncation
    print(df[pii_cols].head(3).to_string())

print("\n2. PROTECTED DATA: Pseudonymized & Anonymized\n")

# Search for columns containing specific keywords to avoid hardcoding exact names
behavioral_cols = [col for col in df.columns if any(kw in col.lower() for kw in ['spend', 'behavior', 'transaction', 'purchase'])]
audit_cols = [col for col in df.columns if any(kw in col.lower() for kw in ['model', 'version', 'rationale', 'audit'])]
reviewer_cols = [col for col in df.columns if any(kw in col.lower() for kw in ['human', 'review', 'approver', 'employee'])]

# Minimization: Drop direct identifiers we don't need
target_drops = [col for col in df.columns if any(kw in col.lower() for kw in ['name', 'email', 'ip'])] + behavioral_cols
df_protected = df.drop(columns=target_drops)

# Pseudonymization: Mask the SSN dynamically
ssn_col = next((col for col in df.columns if 'ssn' in col.lower()), None)
if ssn_col and ssn_col in df_protected.columns:
    df_protected[ssn_col] = df_protected[ssn_col].apply(
        lambda x: '***-**-' + str(x)[-4:] if pd.notnull(x) else x
    )

# Anonymization/Generalization: Extract only the birth year and remove the .0 decimal
dob_col = next((col for col in df.columns if 'birth' in col.lower() or 'dob' in col.lower()), None)
if dob_col and dob_col in df_protected.columns:
    # Convert to datetime, extract year, and force it to the 'Int64' whole number format
    df_protected[dob_col] = pd.to_datetime(df_protected[dob_col], errors='coerce').dt.year.astype('Int64')

safe_cols = [col for col in df_protected.columns if 'ssn' in col.lower() or 'birth' in col.lower() or 'dob' in col.lower()]
if safe_cols:
    print(df_protected[safe_cols].head(3).to_string())


print("\n3. ADVANCED GOVERNANCE GAPS\n")

# Check for Audit Trail
if not audit_cols:
    print("FAILED: No audit trail for decisions (Missing model version tracking or rationale)")
else:
    print(f"PASSED: Audit trail fields found -> {audit_cols}")

# Check for Human Oversight
if not reviewer_cols:
    print("FAILED: Lack of human oversight documentation (No record of human review for algorithmic decisions)")
else:
    print(f"PASSED: Potential human oversight fields found -> {reviewer_cols}")

# Check for Sensitive Behavioral Data
if behavioral_cols:
    print(f"WARNING: Sensitive behavioral data collection detected -> {behavioral_cols}")
else:
    print("PASSED: No obvious sensitive behavioral tracking arrays detected.")

1. ORIGINAL DATA: Exposed Personally Identifiable Information (PII)

IDENTIFIED PII FIELDS: ['applicant_info.full_name', 'applicant_info.email', 'applicant_info.ssn', 'applicant_info.ip_address', 'applicant_info.date_of_birth', 'applicant_info.zip_code']

  applicant_info.full_name       applicant_info.email applicant_info.ssn applicant_info.ip_address applicant_info.date_of_birth applicant_info.zip_code
0              Jerry Smith  jerry.smith17@hotmail.com        596-64-4340            192.168.48.155                   2001-03-09                   10036
1           Brandon Walker  brandon.walker2@yahoo.com        425-69-4784              10.1.102.112                   1992-03-31                   10032
2              Scott Moore     scott.moore94@mail.com        370-78-5178            10.240.193.250                   1989-10-24                   10075

2. PROTECTED DATA: Pseudonymized & Anonymized

  applicant_info.ssn  applicant_info.date_of_birth
0        ***-**-4340                 

# 1. PII Identification & Privacy Demonstration

### 1.1 The Privacy Risk: Identifying PII in NovaCred's Data

Before training a model, a governance audit must be performed to identify Personally Identifiable Information (PII). Training models on raw, unprotected PII exposes the company to severe regulatory penalties under the GDPR and violates the trust of the applicants.

Upon auditing the `raw_credit_applications.json` dataset, we identified the following sensitive fields:
* **Direct Identifiers:** `full_name`, `email`, `ssn`, `ip_address`, and `date_of_birth`. 
* **Sensitive Behavioral Data:** The `spending_behavior` array tracks granular transactional habits. Utilizing this for credit scoring is highly intrusive and presents an ethical risk.

**What we did in the code and why:** To ensure our audit doesn't break if the data engineers change column names, we wrote a dynamic scanner (`pii_keywords`) that automatically searches the dataset for variations of these terms (like 'dob' instead of 'date_of_birth'). 

To mitigate these risks before the data reaches the data science team, we programmatically apply three core privacy techniques: **Data Minimization**, **Pseudonymization**, and **Anonymization**.

### 1.2 GDPR Data Minimization

**The Principle:** GDPR Article 5 mandates that organizations only collect and process data that is *strictly necessary* for the specific business purpose. 

A credit risk algorithm does not need an applicant's name, email address, or IP address to mathematically determine their ability to repay a loan. To enforce Data Minimization, the Python code uses a list comprehension to locate any columns containing 'name', 'email', or 'ip', and permanently removes them using the `df.drop()` function. This ensures the model cannot memorize or leak direct identifiers.

### 1.3 Pseudonymization

**The Principle:** Pseudonymization is the process of replacing or masking private identifiers with artificial identifiers or asterisks. Pseudonymized data can still be re-identified by authorized personnel who hold a secure "key". 

But, we cannot simply delete the Social Security Number (`ssn`) entirely, as NovaCred's internal banking systems likely require a unique ID to track the application's lifecycle. Instead, our code locates the SSN column and applies a `lambda` function to mask all but the last four digits (transforming `596-64-4340` into `***-**-4340`). This allows the core banking system to uniquely identify the applicant while ensuring the AI model only interacts with protected records.

### 1.4 Data Anonymization (Generalization)

**The Principle:** Anonymization alters data so that it can never be reversed to identify the individual. Generalization reduces the granularity of the data to protect privacy while maintaining analytical usefulness.

*While the data science team needs the applicant's age to test for demographic fairness, they do not need the exact birthday, which could be used to re-identify them. Our code finds the date of birth column, converts it into a pandas `datetime` object, and extracts only the year using `.dt.year`. We then cast it to an integer (`.astype('Int64')`) to remove unnecessary decimals. This protects the user's identity while preserving the data's utility for fairness auditing.

# 2. GDPR Compliance Mapping

In [23]:
with open('../data/raw_credit_applications.json', 'r') as f:
    data = json.load(f)

df = pd.json_normalize(data)

print("GDPR COMPLIANCE AUDIT: Automated Gap Analysis\n")

# --- EXPANDED DYNAMIC COLUMN DISCOVERY ---
# Article 5: Storage Limitation (Retention)
retention_keywords = ['retention', 'delete_on', 'expire', 'ttl', 'valid_until']
retention_cols = [col for col in df.columns if any(kw in col.lower() for kw in retention_keywords)]

# Article 6: Lawful Basis (Consent)
consent_keywords = ['consent', 'agreement', 'opt_in', 'gdpr_consent', 'accepted_terms']
consent_cols = [col for col in df.columns if any(kw in col.lower() for kw in consent_keywords)]

# Article 14: Transparency (Data Sources)
source_keywords = ['source', 'origin', 'provider', 'vendor', 'collected_from']
source_cols = [col for col in df.columns if any(kw in col.lower() for kw in source_keywords)]

# Article 17: Right to Erasure
erasure_keywords = ['erasure', 'delete_request', 'forgotten', 'is_deleted', 'deletion_flag', 'remove_request']
erasure_cols = [col for col in df.columns if any(kw in col.lower() for kw in erasure_keywords)]

# Check for Article 5: Storage Limitation (Retention)
if not retention_cols:
    print("FAILED: No retention schedule found (e.g., 'retention_until' is missing).")
    print(" -> Violates GDPR Article 5 (Storage Limitation). Risk of indefinite data retention.\n")
else:
    print(f"PASSED: Retention fields found -> {retention_cols}\n")

# Check for Article 6: Lawful Basis (Consent)
if not consent_cols:
    print("FAILED: No lawful basis logged (e.g., 'consent_timestamp' is missing).")
    print(" -> Violates GDPR Article 6 (Lawful Basis for Processing). No proof of applicant consent.\n")
else:
    print(f"PASSED: Consent fields found -> {consent_cols}\n")
    
# Check for Article 14: Transparency (Data Lineage)
if not source_cols:
    print("FAILED: No data lineage tracking (e.g., 'data_source' is missing).")
    print(" -> Violates GDPR Article 14 (Transparency). Cannot trace the origin of financial records.\n")
else:
    print(f"PASSED: Data source fields found -> {source_cols}\n")

# Check for Article 17: Right to Erasure
if not erasure_cols:
    print("FAILED: No Right to Erasure mechanism detected (e.g., 'is_deleted' flag is missing).")
    print(" -> Violates GDPR Article 17 (Right to Erasure). The schema lacks flags to process and execute permanent hard-deletion requests.\n")
else:
    print(f"PASSED: Erasure request fields found -> {erasure_cols}\n")

GDPR COMPLIANCE AUDIT: Automated Gap Analysis

FAILED: No retention schedule found (e.g., 'retention_until' is missing).
 -> Violates GDPR Article 5 (Storage Limitation). Risk of indefinite data retention.

FAILED: No lawful basis logged (e.g., 'consent_timestamp' is missing).
 -> Violates GDPR Article 6 (Lawful Basis for Processing). No proof of applicant consent.

FAILED: No data lineage tracking (e.g., 'data_source' is missing).
 -> Violates GDPR Article 14 (Transparency). Cannot trace the origin of financial records.

FAILED: No Right to Erasure mechanism detected (e.g., 'is_deleted' flag is missing).
 -> Violates GDPR Article 17 (Right to Erasure). The schema lacks flags to process and execute permanent hard-deletion requests.



### 2.1 Violation of Article 5: Storage Limitation

The Rule: Personal data must be kept in a form which permits identification of data subjects for no longer than is necessary.

The Finding: Our automated scan confirmed that the database schema lacks any retention scheduling fields, even after dynamically checking for common engineering aliases (e.g., `retention_until`, `expire`, `ttl`, or `valid_until`).

The Risk: Storing historical rejected applications indefinitely without automated deletion policies creates an illegal "data swamp" and maximizes exposure in the event of a data breach.

### 2.2 Violation of Article 6: Lawful Basis for Processing

The Rule: Organizations cannot legally process personal data without a lawful basis, typically explicit consent when profiling users for automated decision-making.

The Finding: The automated audit revealed a complete absence of consent-tracking mechanisms across all expected naming conventions (missing `consent`, `opt_in`, `gdpr_consent`, and `accepted_terms` fields).

The Risk: Without timestamped proof of consent, NovaCred has no legal authority to ingest this application data into a machine learning pipeline.

### 2.3 Violation of Article 14: Transparency (Data Sources)

The Rule: If data is not obtained directly from the user (e.g., pulling credit scores from an external bureau), the company must provide the data subject with information about the data source.

The Finding: The dataset fails the lineage scan. It is completely missing crucial tracking fields that identify where external data was acquired (missing `data_source`, `origin`, `provider`, or `vendor`).

The Risk: Without clear data lineage, it is impossible to audit the origin of the financial aggregates or notify users of where their external data was sourced from, severely violating transparency mandates.

### 2.4 Implications for Article 17: Right to Erasure

The Rule: Users have the right to request the permanent deletion of their personal data.

The Finding: Our automated scan revealed that the schema entirely lacks an erasure mechanism, missing critical flags required to process deletion requests (such as `is_deleted`, `deletion_flag`, or `remove_request`). 

The Risk: Because our privacy demonstration utilized pseudonymization rather than irreversible anonymization, the records can still theoretically be linked back to the applicant. Without a programmatic way to flag and process deletion requests, NovaCred cannot technically comply when a user exercises their Right to be Forgotten.

# 3. EU AI Act Implications

The deployment of algorithmic models for credit decisions is not just a technical challenge; it is heavily regulated under the new EU AI Act.

In [24]:
print("EU AI ACT AUDIT: High-Risk System Gaps\n")

# --- EXPANDED DYNAMIC COLUMN DISCOVERY ---
# Traceability: Looking for fields related to model versioning
model_keywords = ['model', 'version', 'algorithm', 'ai_id', 'system_version']
model_version_cols = [col for col in df.columns if any(kw in col.lower() for kw in model_keywords)]

# Transparency: Looking for fields related to decision explainability
rationale_keywords = ['rationale', 'explanation', 'reason', 'justification', 'feature_importance']
rationale_cols = [col for col in df.columns if any(kw in col.lower() for kw in rationale_keywords)]

# Oversight: Looking for fields related to human-in-the-loop review
oversight_keywords = ['human', 'reviewer', 'approver', 'override', 'reviewed_by', 'manual_check']
oversight_cols = [col for col in df.columns if any(kw in col.lower() for kw in oversight_keywords)]

# Check for Audit Trail (Traceability)
if not model_version_cols:
    print("FAILED: No model version tracking found (e.g., 'model_version' or 'algorithm_id' is missing).")
    print(" -> Violates AI Act Traceability requirements. Cannot identify which specific model made the decision.\n")
else:
    print(f"PASSED: Model version fields found -> {model_version_cols}\n")

# Check for Transparency (Explainability)
if not rationale_cols:
    print("FAILED: No decision rationale logged (e.g., 'rejection_reason' or 'explanation' is missing).")
    print(" -> Violates AI Act Transparency requirements. No mathematical explanation for the outcome.\n")
else:
    print(f"PASSED: Decision rationale fields found -> {rationale_cols}\n")

# Check for Human-in-the-loop (Oversight)
if not oversight_cols:
    print("FAILED: No human oversight tracking detected (e.g., 'human_reviewer_id' or 'reviewed_by' is missing).")
    print(" -> Violates AI Act Human Oversight mandates. No proof of human intervention for algorithmic rejections.\n")
else:
    print(f"PASSED: Human oversight fields found -> {oversight_cols}\n")

EU AI ACT AUDIT: High-Risk System Gaps

FAILED: No model version tracking found (e.g., 'model_version' or 'algorithm_id' is missing).
 -> Violates AI Act Traceability requirements. Cannot identify which specific model made the decision.

PASSED: Decision rationale fields found -> ['decision.rejection_reason']

FAILED: No human oversight tracking detected (e.g., 'human_reviewer_id' or 'reviewed_by' is missing).
 -> Violates AI Act Human Oversight mandates. No proof of human intervention for algorithmic rejections.



### 3.1 High-Risk System Classification

The Rule: Under the EU AI Act, any AI system used to evaluate the creditworthiness of natural persons or establish their credit score is explicitly classified as a High-Risk AI System.

The Implication: NovaCred cannot treat this algorithm as a standard internal tool. High-Risk systems are legally required to undergo rigorous pre-deployment conformity assessments, continuous risk management, and strict fairness auditing.

### 3.2 The Legal Reality of Disparate Impact

The Finding: Our prior Data Science audit revealed a Disparate Impact (DI) ratio below 0.8, indicating statistically significant gender-based disparities in loan approval rates.

The Implication: Under the AI Act, deploying a model that exhibits unchecked demographic bias exposes the organization to massive regulatory fines. The bias identified is a critical compliance failure that must be mathematically mitigated before the model can be legally utilized.

### 3.3 Mandatory Human Oversight

The Rule: The AI Act mandates that High-Risk systems must be designed to allow for effective human oversight to prevent automation bias and protect fundamental rights.

The Finding: Our automated scan confirmed that the database schema lacks any fields dedicated to human intervention, even after checking for expanded engineering aliases (missing `human_reviewer_id`, `reviewed_by`, `approver`, or `manual_check` flags).

The Implication: NovaCred must implement a "Human-in-the-loop" protocol. Any application flagged for rejection must be routed to a human loan officer for final review, and that officer's ID must be logged in the database to prove regulatory oversight.

### 3.4 Traceability and Auditability

The Rule: High-Risk systems must automatically generate logs (audit trails) to ensure traceability of the system's functioning throughout its lifecycle.

The Finding: The automated audit revealed a partial failure in traceability metadata. While the schema successfully logs an explanation for the outcome (via the discovered `decision.rejection_reason` field), it completely fails to track which specific version of the algorithm made the decision, missing all expected tracking aliases (e.g., `model_version`, `ai_id`, or `system_version`).

The Implication: Even with the rejection reason logged, without knowing the exact model version that generated it, NovaCred cannot fully guarantee the reproducibility of its decisions. This still violates the AI Act's strict traceability requirements.

# 4. Actionable Governance Controls & Privacy Pipeline

Identifying regulatory gaps is only the first step of data governance; the ultimate goal is to architect compliant, privacy-preserving systems. To mitigate the severe GDPR and EU AI Act violations identified in our audit, NovaCred must implement structural controls (Privacy by Design). Here, we will go through all 502 raw applications, showing how our new governance rules should be implemented, dynamically masking sensitive PII, and outputing a new `compliant_credit_applications.json` dataset, having in mind that this dataset is not supposed to be used, but its only puropse is to serve as a tool to demonstrate how our actions are supposed to work.

In [25]:
import datetime
import json
import copy
import re  # re to safely extract years from messy date formats

print("\nARCHITECTING COMPLIANCE: Generating a Governance-Compliant Dataset\n")

# Load the raw data again to ensure we have the full dataset in memory
with open('../data/raw_credit_applications.json', 'r') as f:
    data = json.load(f)

compliant_dataset = []

# Loop through every single record in the raw data
for record in data:
    # Create a deep copy so we safely modify the structure
    safe_record = copy.deepcopy(record)
    
    # --- 1. Apply Data Minimization, Pseudonymization & Anonymization ---
    if 'applicant_info' in safe_record:
        # Drop direct identifiers (Data Minimization)
        safe_record['applicant_info'].pop('full_name', None)
        safe_record['applicant_info'].pop('email', None)
        safe_record['applicant_info'].pop('ip_address', None)
        
        # Pseudonymize SSN
        if 'ssn' in safe_record['applicant_info']:
            raw_ssn = str(safe_record['applicant_info']['ssn'])
            if raw_ssn and raw_ssn != 'None':
                safe_record['applicant_info']['ssn'] = '***-**-' + raw_ssn[-4:]
                
        # Anonymize Date of Birth (Robust Generalization to Year)
        if 'date_of_birth' in safe_record['applicant_info']:
            raw_dob = str(safe_record['applicant_info']['date_of_birth'])
            # Use RegEx to search for a 4-digit sequence anywhere in the date string
            year_match = re.search(r'\d{4}', raw_dob)
            if year_match:
                safe_record['applicant_info']['date_of_birth'] = year_match.group(0)
            else:
                # Fallback in case a date is completely missing or corrupted
                safe_record['applicant_info']['date_of_birth'] = "Unknown"
            
    # Remove intrusive behavioral data
    if 'spending_behavior' in safe_record:
        del safe_record['spending_behavior']

    # --- 2. Add Missing GDPR Compliance Fields ---
    safe_record['governance_metadata'] = {
        "consent_timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "data_source": "NovaCred_Web_Portal_v2", # Resolves Article 14 Lineage
        "retention_until": (datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(days=180)).isoformat(),
        "is_deleted": False # Resolves Article 17 Erasure tracking
    }

    # --- 3. Add Missing EU AI Act Compliance Fields ---
    if 'decision' not in safe_record:
        safe_record['decision'] = {}

    # Keep existing rejection_reason if it exists, otherwise add a placeholder
    if 'rejection_reason' not in safe_record['decision']:
         safe_record['decision']['rejection_reason'] = "Debt-to-income ratio exceeds threshold."

    safe_record['decision']['audit_trail'] = {
        "model_version": "v2.1.0-fairness-adjusted", # Resolves Traceability
        "requires_human_review": True,               # Enforces Human-in-the-loop
        "human_reviewer_id": "PENDING_ASSIGNMENT"    # Resolves Oversight tracking          
    }
    
    compliant_dataset.append(safe_record)

# --- 4. Save the Output ---
output_path = '../data/compliant_credit_applications_demo.json'
with open(output_path, 'w') as out_file:
    json.dump(compliant_dataset, out_file, indent=2)

print(f"SUCCESS: Transformed {len(compliant_dataset)} records.")
print(f"Fully compliant dataset saved to: {output_path}\n")

# Print one sample record to show the audience the final schema
print("SAMPLE COMPLIANT RECORD:")
print(json.dumps(compliant_dataset[0], indent=2))


ARCHITECTING COMPLIANCE: Generating a Governance-Compliant Dataset

SUCCESS: Transformed 502 records.
Fully compliant dataset saved to: ../data/compliant_credit_applications_demo.json

SAMPLE COMPLIANT RECORD:
{
  "_id": "app_200",
  "applicant_info": {
    "ssn": "***-**-4340",
    "gender": "Male",
    "date_of_birth": "2001",
    "zip_code": "10036"
  },
  "financials": {
    "annual_income": 73000,
    "credit_history_months": 23,
    "debt_to_income": 0.2,
    "savings_balance": 31212
  },
  "decision": {
    "loan_approved": false,
    "rejection_reason": "algorithm_risk_score",
    "audit_trail": {
      "model_version": "v2.1.0-fairness-adjusted",
      "requires_human_review": true,
      "human_reviewer_id": "PENDING_ASSIGNMENT"
    }
  },
  "processing_timestamp": "2024-01-15T00:00:00Z",
  "governance_metadata": {
    "consent_timestamp": "2026-03-06T00:20:36.939797+00:00",
    "data_source": "NovaCred_Web_Portal_v2",
    "retention_until": "2026-09-02T00:20:36.939834+00:00

### 4.1 Implement Cryptographic Consent Mechanisms
**The Fix:** Deploy a mandatory consent gateway at the point of data collection.

**Execution in our Pipeline:** To simulate this enforcement, our code automatically injected a `consent_timestamp` into every record's new `governance_metadata` block, ensuring no data is processed without a timestamped lawful basis.

### 4.2 Automate Data Lifecycle & Retention
**The Fix:** Enforce GDPR storage limitations and Article 17 (Right to Erasure) programmatically.

**Execution in our Pipeline:** The script dynamically calculated and added a `retention_until` timestamp (set to 180 days in the future for rejected applications) and injected an `is_deleted: false` flag to establish a programmatic way to handle future Right to be Forgotten requests.

### 4.3 Algorithmic Audit Trails (Traceability)
**The Fix:** Complete the traceability loop to satisfy the EU AI Act.

**Execution in our Pipeline:** While the schema successfully logged a `rejection_reason`, this is insufficient without knowing the exact model. Our code physically wrote back into the database, tagging every automated decision with `"model_version": "v2.1.0-fairness-adjusted"`.

### 4.4 Mandatory Human-in-the-Loop (Oversight)
**The Fix:** Prevent unchecked algorithmic discrimination and satisfy High-Risk AI mandates.

**Execution in our Pipeline:** The pipeline flagged every rejected application with `"requires_human_review": true` and initialized a `human_reviewer_id` field set to "PENDING_ASSIGNMENT" to guarantee accountability.

### 4.5 Ethical Feature Engineering
**The Fix:** Eliminate proxy discrimination and intrusive behavioral tracking.

**Execution in our Pipeline:** The script aggressively protected user privacy by actively locating and permanently deleting the highly intrusive `spending_behavior` array from all 502 records before saving the final dataset.

### 4.6 Enforce Data Lineage Tracking
**The Fix:** Satisfy GDPR Article 14 Transparency requirements.

**Execution in our Pipeline:** To track the origin of external financial metrics, the code hardcoded `"data_source": "NovaCred_Web_Portal_v2"` into the metadata of every payload, closing the traceability gap.

### 4.7 Automated Data Minimization & Anonymization
**The Fix:** Ensure the Data Science team cannot memorize direct identifiers or reverse-engineer applicant identities.

**Execution in our Pipeline:** The loop programmatically dropped all names, emails, and IPs. It dynamically pseudonymized the Social Security Numbers (leaving only the last 4 digits) and anonymized the dates of birth by extracting only the birth year, perfectly balancing privacy with analytical utility.